## adapted from https://dev.to/botreetechnologies/voice-to-text-using-aws-transcribe-with-python-1cfc


In [ ]:
d = {}

with open("../../../../awskey.csv") as f:
    for l in f:
        name, value = l.split('=')
        d[name] = value.strip()

assert(len(d) == 2)

AWS_ACCESS_KEY_ID = d['AWSAccessKeyId']
AWS_SECRET_ACCESS_KEY = d['AWSSecretKey']

In [ ]:
import boto3
import time
import urllib
import json

job_name = 'my_agassi_job_take_2'

# downloaded from http://www.freeinfosociety.com/media.php?id=3828
job_uri = 'https://s3.amazonaws.com/my-aws-bucket-for-exploration-us-east/agassi_speech.mp3'

transcribe = boto3.client('transcribe', aws_access_key_id=AWS_ACCESS_KEY_ID, aws_secret_access_key=AWS_SECRET_ACCESS_KEY, region_name='us-east-1')

In [ ]:
transcribe.start_transcription_job(TranscriptionJobName=job_name, Media={'MediaFileUri': job_uri}, MediaFormat='mp3', LanguageCode='en-US')

In [ ]:
while True:
    status = transcribe.get_transcription_job(TranscriptionJobName=job_name)
    if status['TranscriptionJob']['TranscriptionJobStatus'] in ['COMPLETED', 'FAILED']:
        break
    print("Not ready yet...")
    time.sleep(2)
print(status)

In [ ]:
import urllib.request

if status['TranscriptionJob']['TranscriptionJobStatus'] == 'COMPLETED':
    response = urllib.request.urlopen(status['TranscriptionJob']['Transcript']['TranscriptFileUri'])
    data = json.loads(response.read())
    text = data['results']['transcripts'][0]['transcript']
    print(text)